# EXP-041 — RealMLP (GPU)

Minimal executor for our own fold-safe pipeline. Enable a Kaggle GPU and Internet (only needed if `pytabkit` is not already available). Add two inputs: the competition data and a small dataset containing `src/train_exp041_realmlp.py`. Prior OOF/test files are optional; if attached anywhere under `/kaggle/input`, the runner uses them for diagnostics and nested blends.

In [ ]:
# Environment report — run before installation/training.
import importlib.util, json, os, platform, sys
import psutil, sklearn, torch
env = {
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none',
    'cuda_available': torch.cuda.is_available(),
    'torch_cuda': torch.version.cuda,
    'vram_gb': torch.cuda.get_device_properties(0).total_memory / 1024**3 if torch.cuda.is_available() else 0,
    'ram_total_gb': psutil.virtual_memory().total / 1024**3,
    'ram_available_gb': psutil.virtual_memory().available / 1024**3,
    'python': platform.python_version(),
    'torch': torch.__version__,
    'sklearn': sklearn.__version__,
    'pytabkit_available': importlib.util.find_spec('pytabkit') is not None,
}
print(json.dumps(env, indent=2))
assert env['cuda_available'], 'EXP-041 is designed for a Kaggle GPU runtime; enable GPU and restart.'

## Minimal installation (separate, conditional)
This installs only `pytabkit`; Kaggle already supplies the CUDA-enabled PyTorch build. It does not install HPO, AutoGluon, or unrelated model extras.

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec('pytabkit') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pytabkit==1.7.3'])
else:
    print('pytabkit already installed; installation skipped')
from pytabkit import RealMLP_TD_Classifier
import importlib.metadata
print('pytabkit', importlib.metadata.version('pytabkit'), '| RealMLP available:', RealMLP_TD_Classifier is not None)

In [ ]:
# Discover inputs by schema/name; no local-machine paths are assumed.
from pathlib import Path
import pandas as pd
input_root = Path('/kaggle/input')
train_candidates = list(input_root.rglob('train.csv'))
data_dir = None
for p in train_candidates:
    try:
        cols = pd.read_csv(p, nrows=2).columns
        if {'id', 'addicted_label', 'daily_screen_time_hours'}.issubset(cols) and (p.parent / 'test.csv').exists():
            data_dir = p.parent
            break
    except Exception:
        pass
scripts = list(input_root.rglob('train_exp041_realmlp.py'))
assert data_dir is not None, 'Competition train.csv/test.csv not found under /kaggle/input'
assert scripts, 'Upload src/train_exp041_realmlp.py as a Kaggle Dataset input'
runner = scripts[0]
print('data_dir:', data_dir)
print('runner:', runner)


In [ ]:
# Execute the complete gated experiment. Prior prediction artifacts are discovered under /kaggle/input.
import subprocess, sys
cmd = [sys.executable, str(runner),
       '--input-dir', str(data_dir),
       '--output-dir', '/kaggle/working',
       '--reference-dir', '/kaggle/input',
       '--device', 'cuda',
       '--max-epochs', '256',
       '--batch-size', '1024']
print(' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# Final inventory and concise metrics. Download exp041_artifacts.zip.
from pathlib import Path
working = Path('/kaggle/working')
for p in sorted(working.glob('exp041*')) + sorted(working.glob('*exp041*')):
    if p.is_file(): print(f'{p.name}: {p.stat().st_size/1024**2:.2f} MB')
metrics = working / 'exp041_realmlp_metrics.txt'
if metrics.exists(): print('\n' + metrics.read_text())
assert (working / 'exp041_artifacts.zip').exists(), 'Artifact zip was not created'